## Tarea 3: ¿Cómo vamos a entrenar la RN? — material de práctica (no calificado)

Esta tarea **ya no se entrega ni se califica**. Es material de práctica y preparación: los exámenes evalúan la teoría detrás de estos experimentos (temarios en `Examenes/`) y el proyecto de colaboración con IA usa este mismo protocolo experimental.

Sugerencias para sacarle provecho:
1. Puedes usar LLMs o agentes de IA libremente — de hecho es buen entrenamiento para el proyecto final. Pero recuerda: los exámenes son en clase y a mano, evalúan que *tú* entiendas.
2. Antes de correr cada experimento, escribe qué esperas observar y por qué; después compara con lo que obtuviste.
3. Asegúrate de entender cada línea del código (tuyo o del agente); consulta la documentación de PyTorch.
4. Mantén el protocolo experimental del curso: semilla fija, mismos splits, un factor variado a la vez, curvas + tabla resumen + interpretación.

### Ejercicios de Python: 

1. Regresión de Poisson: usa la función de acá abajo para generar datos sintéticos de conteos de peatones (número de peatones por minuto en un punto de la ciudad, según hora del día, ubicación y tipo de colonia). Implementa el modelo en Pytorch que prediga $s = f(x,\phi)$ y use $\lambda = e^s$. Entrena con la pérdida de Poisson. Compara entrenar con SGD, SGD con momentum y Adam.

a. Asegurate de entrenar con los mismos parámetros `lr=1e-2, epochs=80, batch_size=64` y `momentum = 0.9`, grafica las curvas de pérdida por época para los tres optimizadores. ¿Qué optimizador converge más rápido y por qué?

b. Ahora usa una tasa de aprendizaje distinta para cada algoritmo `lr_map = {'SGD': 1e-3, 'SGD_momentum': 1e-3, 'Adam': 1e-2}`. Grafica otra vez las curvas de pérdida. ¿Qué optimizador converge más rápido y por qué?

c. ¿Qué esta pasando cuando cambiamos las tasas de aprendizaje?


In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

# Reproducibilidad: fija la semilla
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)
device = torch.device("cpu")  # o "cuda" si tienes GPU


def generate_p1_data(N=2000):
    """
    Datos para P1 (Poisson counts):
     - features: [hour_sin, hour_cos, lat, lon, neigh_onehot(3)]
     - latent s = linear(X)*w + small nonlinear rush-hour term
     - lambda = exp(s), y ~ Poisson(lambda)
    """
    rng = np.random.RandomState(0)
    hours = rng.randint(0, 24, size=(N, 1)).astype(np.float32)
    hour_rad = 2 * math.pi * hours / 24.0
    hour_sin = np.sin(hour_rad).astype(np.float32)
    hour_cos = np.cos(hour_rad).astype(np.float32)
    lat = (40.0 + rng.normal(scale=0.01, size=(N,1))).astype(np.float32)
    lon = (-74.0 + rng.normal(scale=0.01, size=(N,1))).astype(np.float32)
    neigh = rng.randint(0, 3, size=(N,))
    neigh_onehot = np.zeros((N,3), dtype=np.float32)
    neigh_onehot[np.arange(N), neigh] = 1.0

    X = np.concatenate([hour_sin, hour_cos, lat, lon, neigh_onehot], axis=1)
    # pesos "verdaderos" escogidos para producir lambdas razonables
    w = np.array([0.8, -0.6, 30.0, -20.0, 0.5, 0.2, -0.1], dtype=np.float32)
    s_linear = X @ w.reshape(-1,1)
    # horario pico (mañana y tarde)
    rush = 1.0 * np.exp(-((hours - 8)**2)/8.0) + 1.2 * np.exp(-((hours - 18)**2)/10.0)
    s = s_linear + 0.3 * rush
    lam = np.clip(np.exp(s), 1e-6, 50.0)
    y = rng.poisson(lam).astype(np.float32)

    return torch.from_numpy(X).float(), torch.from_numpy(y).float(), {'w_true': w}

P1: X1.shape = torch.Size([2000, 7]) y1.shape = torch.Size([2000, 1])
P2: X2.shape = torch.Size([2000, 5]) Y2.shape = torch.Size([2000, 2])


/var/folders/bl/8nlz54wj6vz1h95x_pcnxjc40000gn/T/ipykernel_22386/1880895205.py:48: RuntimeWarning: overflow encountered in exp
  lam = np.clip(np.exp(s), 1e-6, 50.0)


2. Generamos un dataset sintético con dos salidas: altura (metros) y peso (kilogramos). Entrena y compara dos versiones de la misma red neuronal:
- Modelo A: entrena directamente sobre targets sin normalizar.
- Modelo B: normaliza los targets (z-score) durante el entrenamiento; al evaluar des-normaliza las predicciones para reportar métricas en las unidades reales.

a. Grafica la función de pérdida (MSE) global (train y val) en unidades reales por época para ambos modelos en la misma figura

b. Grafica RMSE por salida (altura, peso) en validación por época para ambos modelos 

c. Tabla con RMSE final en validación (altura y peso) para ambos modelos

d. Explica por qué la normalización de los targets ayuda 

c. ¿Qué harias si quisieras priorizar la precisión en altura sobre el peso? (pista: piensa en la función de pérdia o normaliza de forma distinta)

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

# -----------------------------
# Reproducibilidad: fija la semilla
# -----------------------------
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)
device = torch.device("cpu")  # o "cuda" si tienes GPU

def generate_p2_data(N=2000):
    """
    Datos para P2 (height, weight):
      - X: 5 features N(0,1)
      - Height ~ 1.7 + small linear effect + noise(std ~0.08)
      - Weight ~ 70 + larger linear effect + noise(std ~7)
    """
    rng = np.random.RandomState(1)
    X = rng.normal(size=(N,5)).astype(np.float32)
    w_h = np.array([0.02, -0.01, 0.03, 0.0, 0.01], dtype=np.float32)
    w_w = np.array([1.2, -0.8, 0.5, 0.3, 0.1], dtype=np.float32)
    height = 1.7 + X @ w_h.reshape(-1,1) + rng.normal(scale=0.08, size=(N,1)).astype(np.float32)
    weight = 70.0 + X @ w_w.reshape(-1,1) + rng.normal(scale=7.0, size=(N,1)).astype(np.float32)
    Y = np.concatenate([height, weight], axis=1).astype(np.float32)
    return torch.from_numpy(X).float(), torch.from_numpy(Y).float(), {'w_h': w_h, 'w_w': w_w}

X2, Y2, info2 = generate_p2_data(N=2000)   # P2
print("P2: X2.shape =", X2.shape, "Y2.shape =", Y2.shape)

P1: X1.shape = torch.Size([2000, 7]) y1.shape = torch.Size([2000, 1])
P2: X2.shape = torch.Size([2000, 5]) Y2.shape = torch.Size([2000, 2])


/var/folders/bl/8nlz54wj6vz1h95x_pcnxjc40000gn/T/ipykernel_22386/1880895205.py:48: RuntimeWarning: overflow encountered in exp
  lam = np.clip(np.exp(s), 1e-6, 50.0)
